# Bulan 3: Enrichment & Validasi
## Minggu 9-10 (Domain-Specific Labeling)

Tahap ini bertujuan untuk memberikan konteks pada data yang sudah bersih. Kita menggunakan teknik **Keyword-based Labeling** untuk mengelompokkan data ke dalam kategori tertentu (Misal: kelas keluhan kesehatan mental seperti Insomnia, Depresi, Cemas, dan Stress).

In [ ]:
import pandas as pd
import os

# Memuat dataset hasil Bulan 2 (Gunakan dataset_cleaned jika pipeline akhir belum di-run)
file_path_1 = '../Bulan 2/pipeline_akhir_bulan_2.xlsx'
file_path_2 = '../Bulan 2/dataset_cleaned.xlsx'

if os.path.exists(file_path_1):
    df = pd.read_excel(file_path_1)
    text_col = 'normalized_text' if 'normalized_text' in df.columns else 'cleaned_text'
    print('Memuat data dari pipeline akhir Bulan 2')
elif os.path.exists(file_path_2):
    df = pd.read_excel(file_path_2)
    text_col = 'cleaned_text'
    print('Memuat data dari Basic Cleaning (dataset_cleaned.xlsx)')
else:
    print('Dataset tidak ditemukan! Harap jalankan proses Bulan 2 terlebih dahulu.')
    
print(f'Jumlah data: {len(df)}')
df.head(3)

### 1. Definisikan Keyword dan Kategori (Rules)
Kita menyusun *dictionary* yang berisi daftar kata kunci untuk setiap kategori kelas kesehatan mental.

In [ ]:
# Mendefinisikan aturan keyword untuk setiap kategori
labeling_rules = {
    'Insomnia': ['tidur', 'lelah', 'begadang', 'ngantuk', 'insomnia', 'capek', 'melek'],
    'Depresi': ['sedih', 'nangis', 'menangis', 'putus asa', 'hancur', 'depresi', 'nyerah'],
    'Cemas': ['cemas', 'takut', 'khawatir', 'gugup', 'overthinking', 'panik', 'gelisah'],
    'Stress': ['stres', 'stress', 'pusing', 'muak', 'gila', 'beban', 'pusing', 'berat']
}

def apply_keyword_labeling(text):
    if not isinstance(text, str):
        return 'Tidak Terdefinisi'
    
    text = text.lower()
    
    # Mengecek setiap kategori dan keyword-nya
    for label, keywords in labeling_rules.items():
        for keyword in keywords:
            if keyword in text:
                return label  # Kembalikan label jika ditemukan kecocokan pertama
            
    # Jika tidak ada keyword yang cocok sama sekali
    return 'Lainnya / Netral'

### 2. Mengaplikasikan Labeling ke Dataset

In [ ]:
# Terapkan fungsi ke kolom teks
df['Kategori_Mental'] = df[text_col].apply(apply_keyword_labeling)

# Hitung distribusi label
label_counts = df['Kategori_Mental'].value_counts()
print('Distribusi Kategori Label:')
print(label_counts)

### 3. Evaluasi (Demo Data yang Berhasil Dilabeli)
Menampilkan contoh data yang terdeteksi masuk ke kategori tertentu (bukan Netral).

In [ ]:
pd.set_option('display.max_colwidth', None)
# Ambil sampel yang BUKAN kategori 'Lainnya / Netral'
df_labeled_only = df[df['Kategori_Mental'] != 'Lainnya / Netral']

if len(df_labeled_only) > 0:
    demo_df = df_labeled_only[[text_col, 'Kategori_Mental']].head(10)
    display(demo_df)
else:
    print('Tidak ada data yang mengandung keyword kesehatan mental di dataset ini.')
    # Karena dataset scraping sebelumnya (Knetz vs Indo) mungkin tidak berhubungan dengan mental health,
    # Hasilnya bisa jadi Netral semua. Hal ini wajar secara metode.

In [ ]:
# Menyimpan dataset yang sudah dilabeli
output_file = 'dataset_labeled_bulan3.xlsx'
df.to_excel(output_file, index=False)
print(f'Proses Labeling Selesai! Data disimpan di: {output_file}')